In [1]:
from torchvision.datasets import MNIST, FashionMNIST, CIFAR10
import torchvision
import numpy as np
import random

import torch
import torch.nn.functional as F
import cl_gym as cl

import sys
import os

init_path = os.path.abspath('.')
new_path = init_path
while True:
    if new_path[-3:] == "FSW":
        sys.path.append(new_path)
        break
    new_path = os.path.abspath('..')
    os.chdir(new_path)


seed = 0

np.random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False
torch.set_num_threads(8)

def make_params() -> dict:
    import os
    from pathlib import Path
    import uuid

    params = {
            # dataset
            'dataset': "Bios",
            'fairness_agg': 'mean',
            # 'model': 'MLP',

            # benchmark
            'seed': seed,
            'num_tasks': 5,
            'epochs_per_task': 10,
            'per_task_examples': np.inf,
            # 'per_task_examples': 10000,
            'per_task_memory_examples': 320,
            'batch_size_train': 64,
            'batch_size_memory': 64,
            'batch_size_validation': 256,
            'tau': 10.0,

            # algorithm
            'optimizer': 'sgd',
            'learning_rate': 2e-5,
            'momentum': 0.9,
            'learning_rate_decay': 1.0,
            'criterion': torch.nn.CrossEntropyLoss(),
            # 'criterion': torch.nn.BCEWithLogitsLoss(),

            'device': torch.device('cuda:6' if torch.cuda.is_available() else 'cpu'),
             
            # sample selection
            'alpha': 0.001,
            'metric' : "DP",
            'lambda': 1.0,
            'lambda_old': 0.0,

            # postprocessing
            # "post_processing": "eps_fairness"

              }
    

#     trial_id = str(uuid.uuid4())
    trial_id = f"demo/dataset={params['dataset']}/seed={params['seed']}_epoch={params['epochs_per_task']}_lr={params['learning_rate']}_tau={params['tau']}_alpha={params['alpha']}"
    if params['lambda'] != 0:
        trial_id+=f"_lmbd_{params['lambda']}_lmbdold_{params['lambda_old']}"
    params['trial_id'] = trial_id
    params['output_dir'] = os.path.join("./outputs/{}".format(trial_id))
    print(f"output_dir={params['output_dir']}")
    Path(params['output_dir']).mkdir(parents=True, exist_ok=True)

    return params

params = make_params()

output_dir=./outputs/demo/dataset=Bios/seed=0_epoch=10_lr=2e-05_tau=10.0_alpha=0.001_lmbd_1.0_lmbdold_0.0


In [2]:
"MNIST" in params['dataset']

False

In [3]:
from datasets.bios import Bios

if params['dataset'] in ["Bios"]:
    benchmark = Bios(num_tasks=params['num_tasks'],
                        per_task_memory_examples=params['per_task_memory_examples'],
                        per_task_examples = params['per_task_examples'],
                        max_length = 128,
                        random_class_idx = False)
    input_dim = (12)
    class_idx = benchmark.class_idx
    num_classes = len(class_idx)

In [4]:
from backbones.bert import BertClassifier

backbone = BertClassifier(num_classes, benchmark.bert_config['model'], params)

from trainers import FairContinualTrainer
from trainers.fair_trainer import FairContinualTrainer2 as ContinualTrainer
from metrics import FairMetricCollector
from metrics import MetricCollector2

from algorithms import Heuristic3

algorithm = Heuristic3(backbone, benchmark, params, requires_memory=True)

metric_manager_callback = FairMetricCollector(num_tasks=params['num_tasks'],
                                                        eval_interval='epoch',
                                                        epochs_per_task=params['epochs_per_task'])
# metric_manager_callback = MetricCollector2(num_tasks=params['num_tasks'],
#                                                         eval_interval='epoch',
#                                                         epochs_per_task=params['epochs_per_task'])
# from trainers.baselines import BaseMemoryContinualTrainer as ContinualTrainer
# from trainers.baselines import BaseContinualTrainer as ContinualTrainer

trainer = ContinualTrainer(algorithm, params, callbacks=[metric_manager_callback]) # type: ignore
# 
# trainer = FairContinualTrainer2(algorithm, params, callbacks=[metric_manager_callback])


In [5]:
if params['fairness_agg'] == "mean":
    agg = np.mean
elif params['fairness_agg'] == "max":
    agg = np.max
else:
    raise NotImplementedError

fairness_metrics = ["std", "EER", "EO", "DP"]
for metric in metric_manager_callback.meters:
    if metric in fairness_metrics:
        metric_manager_callback.meters[metric].agg = agg


In [6]:
params['alpha']

0.001

In [7]:
trainer.run()
print("final avg-acc", metric_manager_callback.meters['accuracy'].compute_final())
print("final avg-forget", metric_manager_callback.meters['forgetting'].compute_final())

---------------------------- Task 1 -----------------------
[1] Eval metrics for task 1 >> {'accuracy': 0.6976970955472479, 'loss': 0.0023360830628216733, 'std': 0.3486153692888529, 'EER': -1, 'EO': [0.009858539611530048, 0.0032041488897477377, 0.006431999037552901, 0.19921163396284214, 0.033704788888395676], 'DP': -1, 'accuracy_s0': 0.6709935706348011, 'accuracy_s1': 0.7027688017569454, 'classwise_accuracy': {3: array([5016, 6081]), 0: array([28246, 29527]), 4: array([  54, 4987]), 1: array([8764, 9641]), 2: array([6416, 8151])}, 'DP_ingredients': {'class_pred_count_s0': {3: 4271, 0: 19352, 1: 2650, 2: 5177, 4: 26, 18: 1}, 'class_pred_count_s1': {0: 14076, 1: 7430, 2: 2861, 3: 2506, 4: 37}, 'class_pred_count': {3: 6777, 0: 33428, 1: 10080, 2: 8038, 4: 63, 18: 1}, 'count_s0': 31477, 'count_s1': 26910, 'count': 58387}}
[2] Eval metrics for task 1 >> {'accuracy': 0.885030938224056, 'loss': 0.0011677482028426222, 'std': 0.05321173088812073, 'EER': -1, 'EO': [0.006828657962005824, 0.027901

In [8]:
import copy
task_weight = copy.deepcopy(algorithm.weight_all)

num_bin = 20
np.arange(0+1/num_bin, 1+1/num_bin, 1/num_bin)

def bin(w: np.array, num_bin=20):
    out = dict()
    for r in np.arange(0+1/num_bin, 1+1/num_bin, 1/num_bin):
        r = np.round(r, 2)
        out[r] = np.sum(np.logical_and(w<=r, r-1/num_bin<w))
    out[1/num_bin] += np.sum(w==0)
    kk = list(out.keys())
    for k in kk:
        if out[k] == 0:
            del(out[k])
    return out


binned_weight = dict()
for i, wt in enumerate(task_weight):
    if i==0:
        continue
    print(f"task:{i+1}")
    binned_weight[i+1] = list()
    for we in wt:
        binned_weight[i+1].append({k: bin(we[k]) for k in we})




task:2
task:3
task:4
task:5


In [9]:
binned_weight

{2: [{0: {0.05: 8756, 0.2: 1, 0.25: 1, 0.35: 1, 0.9: 1, 1.0: 14397},
   1: {0.05: 13340, 0.3: 1, 0.75: 1, 0.8: 1, 1.0: 16287}},
  {0: {0.05: 4846, 0.35: 1, 0.8: 1, 1.0: 18309},
   1: {0.05: 7374, 0.1: 1, 0.15: 1, 0.3: 1, 0.65: 1, 0.95: 1, 1.0: 22251}},
  {0: {0.05: 4923, 0.95: 1, 1.0: 18233},
   1: {0.05: 6748, 0.1: 1, 0.55: 2, 0.65: 1, 0.75: 1, 0.95: 1, 1.0: 22876}},
  {0: {0.05: 6172, 0.7: 1, 0.9: 1, 1.0: 16983},
   1: {0.05: 9047, 0.55: 1, 0.7: 1, 0.95: 1, 1.0: 20580}},
  {0: {0.05: 8725, 0.5: 1, 0.65: 1, 1.0: 14430},
   1: {0.05: 10222, 0.1: 1, 0.2: 1, 0.55: 1, 1.0: 19405}},
  {0: {0.05: 9334, 0.55: 1, 0.6: 1, 1.0: 13821},
   1: {0.05: 13415, 0.25: 1, 0.3: 1, 0.45: 2, 0.5: 1, 1.0: 16210}},
  {0: {0.05: 10141, 0.45: 1, 0.7: 1, 0.75: 1, 0.85: 1, 1.0: 13012},
   1: {0.05: 14090, 1.0: 15540}},
  {0: {0.05: 12311, 0.7: 1, 1.0: 10845},
   1: {0.05: 17491, 0.15: 2, 0.45: 1, 0.75: 1, 0.8: 1, 0.9: 1, 1.0: 12133}},
  {0: {0.05: 10277, 0.15: 1, 0.4: 1, 0.45: 1, 1.0: 12877},
   1: {0.05: 15833

In [10]:
metric_manager_callback.meters['accuracy'].get_data()

array([[0.917, 0.   , 0.   , 0.   , 0.   ],
       [0.759, 0.853, 0.   , 0.   , 0.   ],
       [0.714, 0.794, 0.844, 0.   , 0.   ],
       [0.75 , 0.774, 0.752, 0.872, 0.   ],
       [0.718, 0.724, 0.73 , 0.892, 0.664]])

In [11]:
np.mean(metric_manager_callback.meters['accuracy'].compute_overall())

0.8079090117236254

In [12]:
[np.round(x, 3) for x in metric_manager_callback.meters['EO'].compute_overall()]

[0.035, 0.072, 0.071, 0.082, 0.081]

In [13]:
np.mean(metric_manager_callback.meters['EO'].compute_overall())

0.06825061107121237

In [14]:
[np.round(x, 3) for x in metric_manager_callback.meters['DP'].compute_overall()]

[0.037, 0.028, 0.018, 0.015, 0.011]

In [15]:
np.mean(metric_manager_callback.meters['DP'].compute_overall())

0.02182815390947449